# Module 2.4: Handling Missing Telemetry & Forward Fill
### 🇵🇱 Uzupełnianie Braków w Szeregach Czasowych

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/max480425/industrial-ml-course/blob/main/public/notebooks/02_04_missing_telemetry.ipynb)

**Industrial Machine Learning & Predictive Maintenance Interactive Laboratory**

> **Summary:** Manage industrial network drops, wireless packet losses, and intermittent sensor polling using forward fill (LOCF) and interpolation.  
> 🇵🇱 **Streszczenie:** Radź sobie z przerwami w transmisji sieciowej i gubieniem pakietów za pomocą forward fill oraz interpolacji.

- **Estimated Time:** 20 minutes
- **Runtime:** Python 3.10+ (CPU or Free Tier T4 GPU)
- **Packages:** `numpy`, `scipy`, `scikit-learn`, `pandas`, `matplotlib`, `seaborn`
---


In [ ]:
# Step 1: Install & Verify Python Scientific Environment
!pip install -q numpy scipy scikit-learn pandas matplotlib seaborn

import numpy as np
import pandas as pd
import scipy.signal as signal
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'monospace'
print("✓ Industrial ML environment loaded successfully.")


### Key Engineering Concepts:
1. **Statistical Descriptors**: RMS (destructive energy), Kurtosis (peakedness/impact shocks), and Skewness indicate machine degradation.
2. **StandardScaler Breakdown**: Single extreme spikes (e.g. 5,000A startup surge) artificially inflate variance, crushing normal operational readings into a near-zero range.
3. **RobustScaler**: Uses median and IQR ($Q_3 - Q_1$), providing immunity to severe outliers.


In [ ]:
# 1. Simulate Industrial Motor Current with Outliers
np.random.seed(101)
n_samples = 500

# Normal steady-state operation: Mean = 24.5 A, Std = 2.8 A
normal_current = np.random.normal(loc=24.5, scale=2.8, size=n_samples)

# Contaminate with severe startup surge outliers (300A - 800A)
outlier_indices = np.random.choice(n_samples, size=15, replace=False)
telemetry = normal_current.copy()
telemetry[outlier_indices] = np.random.uniform(250.0, 750.0, size=len(outlier_indices))

df_telemetry = pd.DataFrame({'motor_current_amps': telemetry})
print(df_telemetry.describe().round(2))


In [ ]:
# 2. Compare StandardScaler vs RobustScaler
std_scaler = StandardScaler()
robust_scaler = RobustScaler()

df_telemetry['standard_scaled'] = std_scaler.fit_transform(df_telemetry[['motor_current_amps']])
df_telemetry['robust_scaled'] = robust_scaler.fit_transform(df_telemetry[['motor_current_amps']])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Normal range distribution comparison (excluding extreme outliers for zoom)
normal_mask = ~df_telemetry.index.isin(outlier_indices)

axes[0].hist(df_telemetry.loc[normal_mask, 'standard_scaled'], bins=25, color='#f59e0b', alpha=0.8, edgecolor='black')
axes[0].set_title("StandardScaler on Normal Data (Crushed Resolution)", fontweight='bold')
axes[0].set_xlabel("Scaled Value (Z-Score)")

axes[1].hist(df_telemetry.loc[normal_mask, 'robust_scaled'], bins=25, color='#10b981', alpha=0.8, edgecolor='black')
axes[1].set_title("RobustScaler on Normal Data (Preserved Variance)", fontweight='bold')
axes[1].set_xlabel("Scaled Value (IQR Units)")

plt.tight_layout()
plt.show()


## 📋 Practical Hands-On Checklist

- [ ] **Task 1:** Analyze missing value patterns (random vs burst outages) *(🇵🇱 Zbadaj rozkład braków danych (losowe vs ciągłe awarie sieci))*
- [ ] **Task 2:** Apply forward-fill (LOCF) on discrete valve state signals *(🇵🇱 Zastosuj uzupełnianie w przód (LOCF) dla stanów dyskretnych zaworów)*
- [ ] **Task 3:** Implement cubic spline interpolation for continuous pressure curves *(🇵🇱 Wdróż interpolację sklejaną dla ciągłych krzywych ciśnienia)*


In [ ]:
# Exercise Challenge:
# Compute the Interquartile Range (IQR) for motor_current_amps and count Tukey outliers
q1 = df_telemetry['motor_current_amps'].quantile(0.25)
q3 = df_telemetry['motor_current_amps'].quantile(0.75)
iqr = q3 - q1
outlier_mask = (df_telemetry['motor_current_amps'] < q1 - 1.5 * iqr) | (df_telemetry['motor_current_amps'] > q3 + 1.5 * iqr)
print(f"Identified {outlier_mask.sum()} outliers using Tukey IQR rule (Expected ~15).")
assert outlier_mask.sum() >= 15, "Outlier detection failed to identify injected spikes."
print("✓ Outlier filtering exercise verified.")
